# Unit 11 — FSAs and Regular Expressions Solutions

Worked answers for the short-answer exercises, and reference programs for the programming exercises.
Each programming solution is also saved as a runnable program in the `assets` folder (for example `assets/ex4.py`); run it with `python assets/ex4.py < input.txt`.
Each short-answer answer is checked by a check cell that runs the same FSA or regular expression through a small evaluator kept in `assets/verify/fsa_eval.py`.

## Exercise 1

Trace each string from `q0`.

- `ab`: `q0 → q1 → q2`. It ends in `q2`, the final state, so it is accepted.
- `abb`: `q0 → q1 → q2 → q0`. It ends in `q0`, so it is rejected.
- `bbab`: `q0 → q0 → q0 → q1 → q2`. Accepted.
- `aabab`: `q0 → q1 → q1 → q2 → q1 → q2`. Accepted.
- `ba`: `q0 → q0 → q1`. Rejected.

Every `a` leads to `q1`, and only `b` straight after an `a` reaches `q2`: the FSA accepts exactly the strings that end in `ab`.

**Answer:** `A, C, D`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

table = {
    ("q0", "a"): "q1", ("q0", "b"): "q0",
    ("q1", "a"): "q1", ("q1", "b"): "q2",
    ("q2", "a"): "q1", ("q2", "b"): "q0",
}
options = {"A": "ab", "B": "abb", "C": "bbab", "D": "aabab", "E": "ba"}
chosen = [label for label, s in options.items() if fsa_eval.run_dfa(table, "q0", {"q2"}, s)]
assert str(", ".join(chosen) or "NONE") == "A, C, D"

## Exercise 2

Read `10110` one symbol at a time, starting in `q0`.

- `1`: `q0 → q1`
- `0`: `q1 → q2`
- `1`: `q2 → q2`
- `1`: `q2 → q2`
- `0`: `q2 → q1`

The FSA stops in `q1`.

**Answer:** `q1`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

table = {
    ("q0", "0"): "q0", ("q0", "1"): "q1",
    ("q1", "0"): "q2", ("q1", "1"): "q0",
    ("q2", "0"): "q1", ("q2", "1"): "q2",
}
# The stopping state is the one state that, made the only final state, accepts the string.
ends = [q for q in ["q0", "q1", "q2"] if fsa_eval.run_dfa(table, "q0", {q}, "10110")]
assert len(ends) == 1
assert str(ends[0]) == "q1"

## Exercise 3

From `q0` the only move is `a` to `q1`, and from `q1` the only move is `b` back to `q0`.
So the accepted strings are `ab` repeated any number of times, `(ab)*`, including zero times.

- `λ`: the FSA stays in `q0`, which is final. Accepted.
- `ab`: `q0 → q1 → q0`. Accepted.
- `aba`: ends in `q1`, which is not final. Rejected.
- `abab`: ends in `q0`. Accepted.
- `ba`: `q0` has no move on `b`, so the string is rejected at once.

**Answer:** `A, B, D`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

table = {("q0", "a"): "q1", ("q1", "b"): "q0"}
options = {"A": "", "B": "ab", "C": "aba", "D": "abab", "E": "ba"}
chosen = [label for label, s in options.items() if fsa_eval.run_dfa(table, "q0", {"q0"}, s)]
assert str(", ".join(chosen) or "NONE") == "A, B, D"

## Exercise 4

**Notice:** read the table into a dict keyed by `(state, symbol)`, then walk each string from the initial state.
A missing transition must reject the string, so the program moves to a made-up state `-` that no transition leaves and that is never final (state names are letters and digits, so `-` cannot clash with a real one).
The empty string never enters the loop, so it is accepted exactly when the initial state is final.

In [ ]:
start = input()
finals = input().split()
t = int(input())
table = {}
for k in range(t):
    parts = input().split()
    table[(parts[0], parts[1])] = parts[2]
n = int(input())
for k in range(n):
    word = input()
    if word == "-":
        word = ""
    state = start
    for symbol in word:
        if (state, symbol) in table:
            state = table[(state, symbol)]
        else:
            state = "-"
    if state in finals:
        print("ACCEPT")
    else:
        print("REJECT")

## Exercise 5

Follow the table.
The first symbol must be `a` (`q0 → q1`).
In `q1`, any number of `b`s loop, then one `a` moves to `q2`.
In `q2`, any number of `b`s loop, and there is no move on `a`.
So the language is `a`, then `b*`, then `a`, then `b*`: `ab*ab*`.

Each of the other options accepts a string the FSA rejects:

- **B.** `(ab*)*` accepts `λ`, but `q0` is not final.
- **C.** `ab*a|b*` accepts `λ` (from `b*`), and it does not allow `b`s after the second `a` (the FSA accepts `aab`).
- **D.** `a(b|a)*b*` accepts `a`, which ends in `q1`.

**Answer:** `A`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

table = {
    ("q0", "a"): "q1",
    ("q1", "a"): "q2", ("q1", "b"): "q1",
    ("q2", "b"): "q2",
}
options = {"A": "ab*ab*", "B": "(ab*)*", "C": "ab*a|b*", "D": "a(b|a)*b*"}
chosen = []
for label, pattern in options.items():
    agree = True
    for length in range(8):
        for n in range(2 ** length):
            s = "".join("ab"[(n >> k) & 1] for k in range(length))
            if fsa_eval.accepts(pattern, s) != fsa_eval.run_dfa(table, "q0", {"q2"}, s):
                agree = False
    if agree:
        chosen.append(label)
assert str(", ".join(chosen) or "NONE") == "A"

## Exercise 6

Star binds tightest, then concatenation, then union, so `a|bc*` means `a | (b(c*))`: either `a` alone, or `b` followed by any number of `c`s.

- `a`: the left side. Yes.
- `bccc`: `b` then three `c`s. Yes.
- `acc`: the left side is only `a`, and the right side must start with `b`. No.
- `bcbc`: after `bc` only more `c`s may follow. No.
- `b`: `b` then zero `c`s. Yes.

**Answer:** `A, B, E`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

options = {"A": "a", "B": "bccc", "C": "acc", "D": "bcbc", "E": "b"}
chosen = [label for label, s in options.items() if fsa_eval.accepts("a|bc*", s)]
assert str(", ".join(chosen) or "NONE") == "A, B, E"

## Exercise 7

Union is last, so `(0U1)*00U11*` means `((0U1)*00) U (11*)`:

- the left side is any string of `0`s and `1`s that ends in `00`
- the right side is one or more `1`s

Check each string:

- `100`: ends in `00`. Yes.
- `111`: all `1`s. Yes.
- `0110`: ends in `10`, and it has `0`s. No.
- `00`: ends in `00` (the `(0U1)*` part is empty). Yes.
- `10`: ends in `10`, and it has a `0`. No.

**Answer:** `A, B, D`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

options = {"A": "100", "B": "111", "C": "0110", "D": "00", "E": "10"}
chosen = [label for label, s in options.items() if fsa_eval.accepts("(0U1)*00U11*", s)]
assert str(", ".join(chosen) or "NONE") == "A, B, D"

## Exercise 8

`(a|b)*` is every string of `a`s and `b`s.

- **A.** `(a*b*)*`: each single letter is an `a*b*` block, so any string can be built. Equivalent (identity 7).
- **B.** `a*b*`: all `a`s must come before all `b`s, so `ba` is missing. Not equivalent.
- **C.** `a*(ba*)*`: start with some `a`s, then every `b` brings the `a`s after it. Any string splits this way. Equivalent (identity 8).
- **D.** `(ab)*`: misses `a`. Not equivalent.
- **E.** `(a*|b*)*`: `a` and `b` are each an `a*` or `b*` block, so any string can be built. Equivalent (identity 6).

**Answer:** `A, C, E`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

options = {"A": "(a*b*)*", "B": "a*b*", "C": "a*(ba*)*", "D": "(ab)*", "E": "(a*|b*)*"}
chosen = [label for label, p in options.items() if fsa_eval.same_language(p, "(a|b)*", "ab", 8)]
assert str(", ".join(chosen) or "NONE") == "A, C, E"

## Exercise 9

`a*` is every string of `a`s, including `λ`.

- **A.** `aa*|λ`: one or more `a`s, or the empty string. Equivalent (identity 3).
- **B.** `(a*)*`: repeating a run of `a`s gives a run of `a`s. Equivalent (identity 1).
- **C.** `aa*`: at least one `a`, so `λ` is missing. Not equivalent.
- **D.** `a*a`: also at least one `a`. Not equivalent.
- **E.** `(λ|a)*`: each repetition adds nothing or one `a`, so any number of `a`s, including none. Equivalent.

**Answer:** `A, B, E`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

options = {"A": "aa*|λ", "B": "(a*)*", "C": "aa*", "D": "a*a", "E": "(λ|a)*"}
chosen = [label for label, p in options.items() if fsa_eval.same_language(p, "a*", "ab", 8)]
assert str(", ".join(chosen) or "NONE") == "A, B, E"

## Exercise 10

The string must be one or more of the capitals `A`, `B`, `C`, then at most one small letter, then any number of digits.

- `AB`: capitals `AB`, no small letter, no digits. Yes.
- `CAt77`: capitals `CA`, small letter `t`, digits `77`. Yes.
- `Dog1`: `D` is not in `A`–`C`. No.
- `ABab5`: capitals `AB`, then two small letters `ab`, but only one is allowed. No.
- `B9`: capital `B`, no small letter, digit `9`. Yes.

**Answer:** `A, B, E`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

options = {"A": "AB", "B": "CAt77", "C": "Dog1", "D": "ABab5", "E": "B9"}
chosen = [label for label, s in options.items() if fsa_eval.accepts("[A-C]+[a-z]?[0-9]*", s)]
assert str(", ".join(chosen) or "NONE") == "A, B, E"

## Exercise 11

The string must be `h`, then any one character, then `t`, then any number of characters that are not the vowels `a e i o u`.

- `hat`: `h`, `a`, `t`, and nothing after. Yes.
- `hoot`: the third character is `o`, not `t`. No.
- `hits`: `h`, `i`, `t`, then `s`, which is not a vowel. Yes.
- `h.tea`: `h`, `.`, `t` fits (the `.` in the pattern matches any character, here a dot), but then `e` is a vowel. No.
- `hxtz9`: `h`, `x`, `t`, then `z` and `9`, neither a vowel. Yes.

**Answer:** `A, C, E`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

options = {"A": "hat", "B": "hoot", "C": "hits", "D": "h.tea", "E": "hxtz9"}
chosen = [label for label, s in options.items() if fsa_eval.accepts("h.t[^aeiou]*", s)]
assert str(", ".join(chosen) or "NONE") == "A, C, E"

## Exercise 12

**Notice:** `match(p, i, s, j)` asks whether the pattern from position `i` matches the word from position `j` to its end.
Look one character past the letter for a mark:

- no mark: the letter must match, then move on in both
- `?`: skip the letter, or match it once
- `*`: skip the letter, or match it once and stay on the same pattern item for more
- `+`: match it once, then either move on or stay for more

The base case is the end of the pattern, which matches only the end of the word.

In [ ]:
def match(p, i, s, j):
    if i == len(p):
        return j == len(s)
    fits = j < len(s) and s[j] == p[i]
    mark = ""
    if i + 1 < len(p):
        if p[i + 1] == "?" or p[i + 1] == "+" or p[i + 1] == "*":
            mark = p[i + 1]
    if mark == "":
        return fits and match(p, i + 1, s, j + 1)
    if mark == "?":
        if match(p, i + 2, s, j):
            return True
        return fits and match(p, i + 2, s, j + 1)
    if mark == "*":
        if match(p, i + 2, s, j):
            return True
        return fits and match(p, i, s, j + 1)
    if fits:
        if match(p, i + 2, s, j + 1):
            return True
        return match(p, i, s, j + 1)
    return False


pattern = input()
n = int(input())
for k in range(n):
    word = input()
    if word == "-":
        word = ""
    if match(pattern, 0, word, 0):
        print("ACCEPT")
    else:
        print("REJECT")

## Exercise 13

Track, for each length, how many strings end in each state.
Every state has one move on `0` and one on `1`:

- `q0` goes to `q0` and `q1`
- `q1` goes to `q2` and `q0`
- `q2` goes to `q1` and `q2`

So each state receives, from the counts one step earlier:

- `q0` gets the counts of `q0` and `q1`
- `q1` gets the counts of `q0` and `q2`
- `q2` gets the counts of `q1` and `q2`

| length | q0 | q1 | q2 |
|---|---|---|---|
| 0 | 1 | 0 | 0 |
| 1 | 1 | 1 | 0 |
| 2 | 2 | 1 | 1 |
| 3 | 3 | 3 | 2 |
| 4 | 6 | 5 | 5 |

Six strings of length 4 end in `q0`, the final state.
(This FSA keeps the remainder of the binary number modulo 3: the six are `0000`, `0011`, `0110`, `1001`, `1100` and `1111`, which are 0, 3, 6, 9, 12 and 15.)

**Answer:** `6`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

table = {
    ("q0", "0"): "q0", ("q0", "1"): "q1",
    ("q1", "0"): "q2", ("q1", "1"): "q0",
    ("q2", "0"): "q1", ("q2", "1"): "q2",
}
count = 0
for n in range(2 ** 4):
    s = format(n, "04b")
    if fsa_eval.run_dfa(table, "q0", {"q0"}, s):
        count += 1
assert str(count) == "6"

## Exercise 14

A `b` never changes the state, and an `a` swaps `q0` and `q1`.
So the FSA is in `q0` exactly when it has read an even number of `a`s: it accepts every string with an even number of `a`s (zero included), with any `b`s anywhere.

- **A.** `b*(ab*ab*)*`: some `b`s, then blocks that each hold exactly two `a`s with `b`s anywhere. Every string with an even number of `a`s splits this way. Yes.
- **B.** `(aa|b)*`: the two `a`s of a pair must be next to each other, so `aba` is missing. No.
- **C.** `(ab*a)*`: every block starts with `a`, so `b` is missing. No.
- **D.** `(b|ab*a)*`: single `b`s, or pairs of `a`s with `b`s between them. Every string with an even number of `a`s splits this way. Yes.

**Answer:** `A, D`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

table = {
    ("q0", "a"): "q1", ("q0", "b"): "q0",
    ("q1", "a"): "q0", ("q1", "b"): "q1",
}
options = {"A": "b*(ab*ab*)*", "B": "(aa|b)*", "C": "(ab*a)*", "D": "(b|ab*a)*"}
chosen = []
for label, pattern in options.items():
    agree = True
    for length in range(9):
        for n in range(2 ** length):
            s = "".join("ab"[(n >> k) & 1] for k in range(length))
            if fsa_eval.accepts(pattern, s) != fsa_eval.run_dfa(table, "q0", {"q0"}, s):
                agree = False
    if agree:
        chosen.append(label)
assert str(", ".join(chosen) or "NONE") == "A, D"

## Exercise 15

- **A.** `(ab)*a = a(ba)*`: both sides are `a`, `aba`, `ababa`, …, alternating letters that start and end with `a`. True (identity 5).
- **B.** `(a|b)* = a*|b*`: the left side has `ab`, but the right side has only strings of one letter repeated. False.
- **C.** `a(b|c) = ab|ac`: concatenation distributes over union. True (identity 4).
- **D.** `(a*)*b = a*b`: `(a*)*` is `a*` (identity 1), so both sides are some `a`s then `b`. True.
- **E.** `ab* = (ab)*`: the left side has `abb`, the right side does not. False.

**Answer:** `A, C, D`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

options = {
    "A": ("(ab)*a", "a(ba)*"),
    "B": ("(a|b)*", "a*|b*"),
    "C": ("a(b|c)", "ab|ac"),
    "D": ("(a*)*b", "a*b"),
    "E": ("ab*", "(ab)*"),
}
chosen = [label for label, (p, q) in options.items() if fsa_eval.same_language(p, q, "abc", 6)]
assert str(", ".join(chosen) or "NONE") == "A, C, D"

## Exercise 16

The string must be `ab` one or more times, then at most one `c`, then any number of characters that are not `a`, `b` or `c`.

- `ababx`: `abab`, no `c`, then `x`. Yes.
- `abc`: `ab`, then `c`, then nothing. Yes.
- `abcc`: `ab`, one `c`, but the second `c` is excluded by `[^abc]`. No.
- `aba`: after `ab` comes `a`, which is neither a new `ab` (no `b` follows) nor allowed by `[^abc]`. No.
- `c`: it must start with `ab`. No.
- `abcdd`: `ab`, `c`, then `dd`. Yes.

**Answer:** `A, B, F`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

options = {"A": "ababx", "B": "abc", "C": "abcc", "D": "aba", "E": "c", "F": "abcdd"}
chosen = [label for label, s in options.items() if fsa_eval.accepts("(ab)+c?[^abc]*", s)]
assert str(", ".join(chosen) or "NONE") == "A, B, F"

## Exercise 17

**Notice:** first cut the pattern into items, each a `(spec, mark)` tuple, so that a class like `[^a-ceiou]` becomes one item.
`fits(spec, c)` decides whether one character fits one item: `.` fits anything, a letter or digit fits itself, and a class checks its letters and ranges with `<=`, then flips the result when it starts with `^`.
The recursive `match` is the Pattern Checker's, working on items instead of characters.

In [ ]:
def read_tokens(p):
    tokens = []
    i = 0
    while i < len(p):
        if p[i] == "[":
            j = i
            while p[j] != "]":
                j = j + 1
            spec = p[i:j + 1]
            i = j + 1
        else:
            spec = p[i]
            i = i + 1
        mark = ""
        if i < len(p):
            if p[i] == "?" or p[i] == "+" or p[i] == "*":
                mark = p[i]
                i = i + 1
        tokens.append((spec, mark))
    return tokens


def in_class(body, c):
    k = 0
    while k < len(body):
        if k + 2 < len(body) and body[k + 1] == "-":
            if body[k] <= c and c <= body[k + 2]:
                return True
            k = k + 3
        else:
            if body[k] == c:
                return True
            k = k + 1
    return False


def fits(spec, c):
    if spec == ".":
        return True
    if len(spec) == 1:
        return spec == c
    body = spec[1:len(spec) - 1]
    if body[0] == "^":
        return not in_class(body[1:len(body)], c)
    return in_class(body, c)


def match(tokens, i, s, j):
    if i == len(tokens):
        return j == len(s)
    spec, mark = tokens[i]
    one = j < len(s) and fits(spec, s[j])
    if mark == "":
        return one and match(tokens, i + 1, s, j + 1)
    if mark == "?":
        if match(tokens, i + 1, s, j):
            return True
        return one and match(tokens, i + 1, s, j + 1)
    if mark == "*":
        if match(tokens, i + 1, s, j):
            return True
        return one and match(tokens, i, s, j + 1)
    if one:
        if match(tokens, i + 1, s, j + 1):
            return True
        return match(tokens, i, s, j + 1)
    return False


tokens = read_tokens(input())
n = int(input())
for k in range(n):
    word = input()
    if word == "-":
        word = ""
    if match(tokens, 0, word, 0):
        print("ACCEPT")
    else:
        print("REJECT")

## Exercise 18

The expression describes the strings that contain `aba` somewhere.
In a string of 5 symbols, `aba` can start at position 1, 2 or 3.

- For each starting position, the other 2 symbols are free: `4` strings each, `12` in all.
- Some strings were counted twice. Starting at 1 and 2 would need the second symbol to be both `b` and `a`: impossible. The same holds for 2 and 3. Starting at 1 and 3 needs `ababa`: one string.
- All three at once is impossible.

By inclusion–exclusion, `12 - 1 = 11`.

**Answer:** `11`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import fsa_eval

count = 0
for n in range(2 ** 5):
    s = "".join("ab"[(n >> k) & 1] for k in range(5))
    if fsa_eval.accepts("(a|b)*aba(a|b)*", s):
        count += 1
assert str(count) == "11"